# Subtask 1: Dimensional Aspect Sentiment Regression (DimASR)

-----


### Outline:

- Installation and importation of necessary libraries
Setting up the project parameters.
Running training and evaluation
Before you start:

- It is strongly advised that you use a GPU to speed up training. To do this, go to the "Runtime" menu in Colab, select "Change runtime type" and then in the popup menu, choose "GPU" in the "Hardware accelerator" box.

### NB:

The codes in this notebook are provided to familiarize yourselves with fine-tuning language models for sentiment regression. You may extend and (or) modify as appropriate to obtain competitive performances.

### Languages and Domains:
#### Track A: Subtask 1
- eng_restaurant
- eng_laptop
- jpn_hotel
- jpn_finance
- rus_restaurant
- tat_restaurant
- ukr_restaurant
- zho_restaurant
- zho_laptop
#### Track B: Subtask 1
- deu-stance
- eng-stance
- hau-stance
- kin-stance
- swa-stance
- twi-stance



In [ ]:
import json
from typing import List, Dict
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, DebertaV2TokenizerFast

from scipy.stats import pearsonr
from tqdm import tqdm
import math
import re
import requests


def load_jsonl(filepath: str) -> List[Dict]:
    with open(filepath, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]

def load_jsonl_url(url: str) -> List[Dict]:
    resp = requests.get(url)
    resp.raise_for_status()
    return [json.loads(line) for line in resp.text.splitlines()]

In [ ]:
import requests
import json
import pandas as pd
from sklearn.model_selection import train_test_split

def load_jsonl_url(url: str):
    """Fetches JSONL data from a URL and returns a list of dictionaries."""
    resp = requests.get(url)
    resp.raise_for_status()
    return [json.loads(line) for line in resp.text.splitlines() if line.strip()]

def jsonl_to_df(data):
    """
    Standardizes JSONL structures based on the specific keys:
    - Train: 'Quadruplet'
    - Dev: 'Aspect_VA'
    - Test: 'Aspect'
    """
    if not data:
        return pd.DataFrame()

    first_entry = data[0]

    # TRAIN: Quadruplet
    if 'Quadruplet' in first_entry:
        df = pd.json_normalize(data, 'Quadruplet', ['ID', 'Text'])
        df[['Valence', 'Arousal']] = df['VA'].str.split('#', expand=True).astype(float)
        df = df.drop(columns=['VA', 'Category', 'Opinion'], errors='ignore')

    # DEV: Aspect_VA
    elif 'Aspect_VA' in first_entry:
        df = pd.json_normalize(data, 'Aspect_VA', ['ID', 'Text'])
        df[['Valence', 'Arousal']] = df['VA'].str.split('#', expand=True).astype(float)
        df = df.drop(columns=['VA'], errors='ignore')

    # TEST: Aspect (List of aspects, no labels)
    elif 'Aspect' in first_entry:
        df = pd.DataFrame(data)
        df = df.explode('Aspect')
        df['Valence'] = 0.0
        df['Arousal'] = 0.0

    else:
        raise ValueError("Unknown data structure. Check JSON keys.")

    # Remove duplicates
    cols = ['ID', 'Text', 'Aspect', 'Valence', 'Arousal']
    return df[cols].drop_duplicates(subset=['ID', 'Aspect']).reset_index(drop=True)

# Configuration
subtask = "subtask_1"
task = "task1"
lang = "eng"
domains = ["restaurant", "laptop"]

all_train_dfs = {}
all_dev_dfs = {}
all_predict_dfs = {}

for domain in domains:
    print(f"--- Loading Datasets for Domain: {domain} ---")

    base_url = f"https://raw.githubusercontent.com/DimABSA/DimABSA2026/refs/heads/main/task-dataset/track_a/{subtask}/{lang}/{lang}_{domain}"
    train_url = f"{base_url}_train_alltasks.jsonl"
    dev_url = f"{base_url}_dev_{task}.jsonl"
    test_url = f"{base_url}_test_{task}.jsonl"

    try:
        # Load and process each set
        train_df = jsonl_to_df(load_jsonl_url(train_url))
        dev_df = jsonl_to_df(load_jsonl_url(dev_url))
        test_df = jsonl_to_df(load_jsonl_url(test_url))

        all_train_dfs[domain] = train_df
        all_dev_dfs[domain] = dev_df
        all_predict_dfs[domain] = test_df

        print(f"{domain.capitalize()} Success:")
        print(f"   - Train: {len(train_df)} rows")
        print(f"   - Dev:   {len(dev_df)} rows (Validation)")
        print(f"   - Test:  {len(test_df)} rows (Final Predictions)")

    except Exception as e:
        print(f"Error processing {domain}: {e}")

--- Loading Datasets for Domain: restaurant ---
Restaurant Success:
   - Train: 3107 rows
   - Dev:   340 rows (Validation)
   - Test:  1504 rows (Final Predictions)
--- Loading Datasets for Domain: laptop ---
Laptop Success:
   - Train: 4958 rows
   - Dev:   275 rows (Validation)
   - Test:  1421 rows (Final Predictions)


In [ ]:
for domain in domains:
    print(f"\n{'='*50}")
    print(f" DOMAIN: {domain.upper()}")
    print(f"{'='*50}")

    # Display Training Set
    if domain in all_train_dfs:
        print(f"\n--- {domain.capitalize()} Training Set  ---")
        display(all_train_dfs[domain][['ID', 'Aspect', 'Valence', 'Arousal', 'Text']].head())

    # Display Development Set
    if domain in all_dev_dfs:
        print(f"\n--- {domain.capitalize()} Development Set ---")
        display(all_dev_dfs[domain][['ID', 'Aspect', 'Valence', 'Arousal', 'Text']].head())

    # Display Test/Prediction Set
    if domain in all_predict_dfs:
        print(f"\n--- {domain.capitalize()} Test Set ---")
        display(all_predict_dfs[domain][['ID', 'Aspect', 'Valence', 'Arousal', 'Text']].head())


 DOMAIN: RESTAURANT

--- Restaurant Training Set  ---


,ID,Aspect,Valence,Arousal,Text
0,rest16_quad_dev_1,NULL,6.75,6.38,ca n ' t wait wait for my next visit .
1,rest16_quad_dev_2,sake list,7.83,8.00,"their sake list was extensive , but we were lo..."
2,rest16_quad_dev_2,NULL,5.00,5.00,"their sake list was extensive , but we were lo..."
3,rest16_quad_dev_3,spicy tuna roll,7.50,7.62,the spicy tuna roll was unusually good and the...
4,rest16_quad_dev_3,rock shrimp tempura,8.25,8.38,the spicy tuna roll was unusually good and the...



--- Restaurant Development Set ---


,ID,Aspect,Valence,Arousal,Text
0,rest26_aspect_va_dev_1,diner food,7.25,6.75,Great diner food and breakfast is served all day
1,rest26_aspect_va_dev_1,breakfast,7.25,6.75,Great diner food and breakfast is served all day
2,rest26_aspect_va_dev_2,food,7.50,7.75,It got very crowded but we still received exce...
3,rest26_aspect_va_dev_2,drinks,7.50,7.50,It got very crowded but we still received exce...
4,rest26_aspect_va_dev_2,service,7.75,7.75,It got very crowded but we still received exce...



--- Restaurant Test Set ---


...

,ID,Aspect,Valence,Arousal,Text
0,laptop_quad_dev_1,unit,7.12,7.12,"this unit is ` ` pretty ` ` and stylish , so m..."
1,laptop_quad_dev_2,device,5.50,5.25,for now i ' m okay with upping the experience ...
2,laptop_quad_dev_3,NULL,5.00,5.12,"seems unlikely but whatever , i ' ll go with it ."
3,laptop_quad_dev_4,version,3.30,6.60,this version has been my least favorite versio...
4,laptop_quad_dev_5,track pad,2.50,6.00,- biggest disappointment is the track pad .



--- Laptop Development Set ---


,ID,Aspect,Valence,Arousal,Text
0,lap26_aspect_va_dev_1,touchscreen,7.80,7.60,The touchscreen works very well
1,lap26_aspect_va_dev_2,HP,2.88,6.75,I am so disappointed in HP
2,lap26_aspect_va_dev_3,keyboard,6.88,6.62,The keyboard is big enough to use for real typing
3,lap26_aspect_va_dev_4,screen size,7.25,7.12,I like the screen size
4,lap26_aspect_va_dev_5,Lenovo,7.38,7.38,Lenovo is my favorite brand of computer



--- Laptop Test Set ---


,ID,Aspect,Valence,Arousal,Text
0,lap26_aspect_va_test_1,Dell,0.0,0.0,"Other than that , I consider this Dell produc..."
1,lap26_aspect_va_test_2,processor,0.0,0.0,The processor was really fast especially for t...
2,lap26_aspect_va_test_3,atom processor,0.0,0.0,The quad core atom processor on the transforme...
3,lap26_aspect_va_test_4,fan,0.0,0.0,The fan runs very cool which prevents it from ...
4,lap26_aspect_va_test_5,Lenovo box,0.0,0.0,"It was sent in a Lenovo box , but looked like..."


# Roberta - large Model

In [ ]:
class VADataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len=128):
        self.tokenizer = tokenizer
        self.data = dataframe.reset_index(drop=True)
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]

        text = f"{row['Aspect']}: {row['Text']}"

        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor([row['Valence'], row['Arousal']], dtype=torch.float)
        }

In [ ]:
class RoBERTaVARegressor(nn.Module):
    def __init__(self, model_path="roberta-large"):
        super(RoBERTaVARegressor, self).__init__()
        self.roberta = AutoModel.from_pretrained(model_path)
        self.dropout = nn.Dropout(0.2)
        self.regressor = nn.Linear(self.roberta.config.hidden_size, 2)

    def forward(self, input_ids, attention_mask):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.last_hidden_state[:, 0, :]

        pooled_output = self.dropout(pooled_output)
        return self.regressor(pooled_output)

In [ ]:
def train_epoch(model, data_loader, optimizer, loss_fn, device):
    """
    Performs one full training pass over the dataset.
    """
    model.train()
    total_loss = 0

    for batch in tqdm(data_loader, desc="Training"):
        optimizer.zero_grad()

        ids = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        outputs = model(ids, mask)

        # Calculate MSE loss
        loss = loss_fn(outputs, labels)

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0) # Gradient clipping
        optimizer.step()
        total_loss += loss.item()

    # Return average loss per batch
    return total_loss / len(data_loader)

def eval_epoch(model, data_loader, loss_fn, device):
    """
    Performs one validation pass to monitor model performance.
    """
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for batch in data_loader:
            ids = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            # Forward pass
            outputs = model(ids, mask)
            loss = loss_fn(outputs, labels)
            total_loss += loss.item()

    return total_loss / len(data_loader)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("roberta-large")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
loss_fn = nn.MSELoss()

print(f"Model and training functions ready. Using device: {device}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Model and training functions ready. Using device: cuda


## Training Loop

In [ ]:
import torch
import gc
import torch.optim as optim
from torch.optim import AdamW
from tqdm import tqdm

#Hyperparameters
BATCH_SIZE = 4
EPOCHS = 5
LEARNING_RATE = 1e-5

for domain in domains:
    print(f"\nStarting Training for Domain: {domain.upper()}")

    # Datasets and Loaders
    train_ds = VADataset(all_train_dfs[domain], tokenizer)
    dev_ds = VADataset(all_dev_dfs[domain], tokenizer)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    dev_loader = DataLoader(dev_ds, batch_size=BATCH_SIZE)

    # Initialize Model and Optimizer
    model = RoBERTaVARegressor().to(device)
    optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)
    loss_fn = nn.MSELoss()

    best_val_loss = float('inf')

    for epoch in range(EPOCHS):
        avg_train_loss = train_epoch(model, train_loader, optimizer, loss_fn, device)
        avg_val_loss = eval_epoch(model, dev_loader, loss_fn, device)

        print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

        # Save best model based on validation loss
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), f"best_model_{domain}.pt")
            print(f"Saved best {domain} model weights!")

    del model
    del optimizer
    gc.collect()
    torch.cuda.empty_cache()
    print(f"Training for {domain} complete.")

print("\nAll domain training finished!")


Starting Training for Domain: RESTAURANT


model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Training: 100%|██████████| 777/777 [05:31<00:00,  2.35it/s]


Epoch 1/5 | Train Loss: 2.2533 | Val Loss: 1.1838
Saved best restaurant model weights!


Training: 100%|██████████| 777/777 [05:32<00:00,  2.34it/s]


...

Epoch 3/5 | Train Loss: 0.4877 | Val Loss: 0.5746


Training: 100%|██████████| 1240/1240 [08:49<00:00,  2.34it/s]


Epoch 4/5 | Train Loss: 0.3967 | Val Loss: 0.5387
Saved best laptop model weights!


Training: 100%|██████████| 1240/1240 [08:50<00:00,  2.34it/s]


Epoch 5/5 | Train Loss: 0.3454 | Val Loss: 0.6026
Training for laptop complete.

All domain training finished!


## Overall Performance

In [ ]:
import numpy as np
from sklearn.metrics import mean_squared_error

def get_overall_rmse(domain):
    model = RoBERTaVARegressor().to(device)
    model.load_state_dict(torch.load(f"best_model_{domain}.pt"))
    model.eval()

    dev_loader = DataLoader(VADataset(all_dev_dfs[domain], tokenizer), batch_size=BATCH_SIZE)

    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in dev_loader:
            outputs = model(batch['input_ids'].to(device), batch['attention_mask'].to(device))
            all_preds.append(outputs.cpu().numpy())
            all_labels.append(batch['labels'].numpy())

    rmse = np.sqrt(mean_squared_error(np.vstack(all_labels), np.vstack(all_preds)))
    return rmse

for domain in domains:
    score = get_overall_rmse(domain)
    print(f"Final Dev RMSE for {domain.upper()}: {score:.4f}")

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Final Dev RMSE for RESTAURANT: 0.7431


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Final Dev RMSE for LAPTOP: 0.7346


## Generating Submission File

### Step 5: Save and submit prediction results

- Define helper `df_to_jsonl`:
  - Sort by ID number.
  - Group rows by ID.
  - Save predictions in JSONL format (`ID`, `Aspect_VA`).
- Run the model on the predict sets (laptop & restaurant).
- Fill in predicted Valence/Arousal values.
- Export three JSONL files:




  - `pred_eng_laptop.jsonl`
  - `pred_eng_restaurant.jsonl`
  - `pred_zho_laptop.jsonl`
- These files can be uploaded as the final submission.


### File Naming Guidelines
When submitting your predictions on the Codabench task page:

Decide the target language(s) and domain(s). Each submission file corresponds to one language-domain combination.
For each language-domain combination, name the file pred_[lang_code]_[domain].jsonl, where
- [lang_code] represents a 3-letter language code, and
- [domain] represents a domain.
For example, Hausa predictions for the movie domain should be named pred_hau_movie.jsonl.
If submitting for multiple languages or domains, submit one prediction file per language-domain combination. For example, submitting for multiple languages or domains would look like this:
```plaintext
subtask_1
├── pred_eng_restaurant.jsonl
├── pred_eng_laptop.jsonl
└── pred_zho_laptop.jsonl

In [ ]:
import os
import zipfile
from google.colab import files

output_dir = "subtask_1"
os.makedirs(output_dir, exist_ok=True)

for domain in domains:
    print(f"\nPredicting for TEST set: {domain.upper()}")
    model = RoBERTaVARegressor().to(device)
    model.load_state_dict(torch.load(f"best_model_{domain}.pt"))
    model.eval()

    test_df = all_predict_dfs[domain]
    test_loader = DataLoader(VADataset(test_df, tokenizer), batch_size=BATCH_SIZE)

    preds = []
    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Inference"):
            outputs = model(batch['input_ids'].to(device), batch['attention_mask'].to(device))
            preds.extend(outputs.cpu().numpy())

    # Grouping by ID to match required format
    grouped = {}
    for i, (_, row) in enumerate(test_df.iterrows()):
        v, a = np.clip(preds[i], 1.0, 9.0)
        clean_aspect = str(row['Aspect']).strip()
        entry = {"Aspect": clean_aspect, "VA": f"{v:.2f}#{a:.2f}"}

        if row['ID'] not in grouped:
            grouped[row['ID']] = {"ID": row['ID'], "Aspect_VA": []}

        # duplicate check
        if not any(x['Aspect'] == entry['Aspect'] for x in grouped[row['ID']]["Aspect_VA"]):
            grouped[row['ID']]["Aspect_VA"].append(entry)

    # Save to JSONL
    save_path = f"{output_dir}/pred_eng_{domain}.jsonl"
    with open(save_path, "w", encoding="utf-8") as f:
        for item in grouped.values():
            f.write(json.dumps(item) + "\n")

#Validation & Download
valid_submission = True
for f_name in os.listdir(output_dir):
    with open(f"{output_dir}/{f_name}", 'r') as f:
        for line in f:
            data = json.loads(line)
            if not data.get("ID") or not data.get("Aspect_VA"):
                valid_submission = False
                print(f"Error: Missing data in {f_name}")

if valid_submission:
    zip_name = "submission_final_validated.zip"
    with zipfile.ZipFile(zip_name, 'w') as z:
        for f in os.listdir(output_dir):
            z.write(f"{output_dir}/{f}", f"{output_dir}/{f}")
    print("All checks passed! Downloading file...")
    files.download(zip_name)


Predicting for TEST set: RESTAURANT


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Inference: 100%|██████████| 376/376 [00:40<00:00,  9.39it/s]



Predicting for TEST set: LAPTOP


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Inference: 100%|██████████| 356/356 [00:37<00:00,  9.59it/s]


All checks passed! Downloading file...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>